In [1]:
import pandas as pd
import json
import glob
import os
import re

from functools import reduce

In [2]:
df_int = pd.read_csv("../../../../../dataset/final_dataset/contacted_anon.csv")

# Only keep vacancies with at least 15 interactions
relevant_vacancies = df_int["cvid"].value_counts()[df_int["cvid"].value_counts() >= 15].index

# Filter to only relevant vacancies
df_int = df_int[df_int["cvid"].isin(relevant_vacancies.values)][["humanjobid", "cvid"]]

In [3]:
df_int

,humanjobid,cvid
28,1527549.0,1ea333bcf408489d88d6bb4ecad3852d
53,1527507.0,6f38a6778f1249b5b14658aed49f5348
61,1527507.0,238e44044b5348e09dba220756613ac6
62,1527507.0,6e5a9a5cac7e454bb17c85853850b108
63,1527507.0,2772752bd7fb4fdebd3c2cdbf7539ee7
...,...,...
242096,1596374.0,4f953842b19b4882bd7b7af6e8d081b7
242134,1598296.0,614f8e1593ce4fa9b238f18f7cd9ed45
242135,1598296.0,05cfcebebe594281a6dc6874e852aad1
242137,1598296.0,208fd8f7b17742ce820ca9f12313ddb9


In [4]:
# 1. Get a list of all your JSON files
file_pattern = './bridge_triples_*.json' 
files = glob.glob(file_pattern)

# Step 1: Group all DataFrames by their model/prompt label
grouped_data = {}

for file in files:
    if os.path.getsize(file) == 0:
        continue

    try:
        with open(file, 'r') as f:
            data = json.load(f)
        
        temp_df = pd.DataFrame(data)
        
        # Extract label (e.g., 'qwen unstructured')
        basename = os.path.basename(file)
        name_match = re.search(r'triples_(.*?)_\d', basename)
        label = name_match.group(1).rstrip('_').replace('_', ' ') if name_match else "unknown"
        label = label.replace(" ", "_")
        
        if label not in grouped_data:
            grouped_data[label] = []
        
        grouped_data[label].append(temp_df)

    except Exception as e:
        print(f"Error reading {file}: {e}")

# Step 2: For each label, "squash" multiple files into one high-density DF
final_model_dfs = []

for label, dfs in grouped_data.items():
    # Stack all files for this model vertically
    combined = pd.concat(dfs, ignore_index=True)
    
    # Sort so that if there are duplicates, we have a consistent pick 
    # (Optional: sort by a timestamp if you want the newest values to take priority)
    # combined = combined.sort_values('some_timestamp_column', ascending=False)

    # This fills gaps where one file had IDs 0-500 and another had 500-1000
    squashed = combined.drop_duplicates(subset=['cvid', 'humanjobid'], keep='last')
    
    # Rename to your specific format
    rename_map = {
        'bridge_triples': f'bridges_{label}',
    }
    squashed = squashed.rename(columns=rename_map)
    
    # Keep only the ID and the new model-specific columns
    cols_to_keep = ['cvid', "humanjobid", f'bridges_{label}']
    squashed = squashed[squashed.columns.intersection(cols_to_keep)]
    
    final_model_dfs.append(squashed)

# Step 3: Horizontal merge of the distinct models
if final_model_dfs:
    main_df = df_int
    for next_df in final_model_dfs:
        main_df = pd.merge(main_df, next_df, on=['cvid', 'humanjobid'], how='outer')
    
    main_df = main_df.sort_values('cvid').reset_index(drop=True)
    
    print(f"Final Shape: {main_df.shape}")
    print("\nSample of merged columns:")
    display(main_df.head(5))
else:
    print("No data processed.")

Final Shape: (24057, 11)

Sample of merged columns:


,humanjobid,cvid,bridges_gemma_semi-structured,bridges_gemma_structured,bridges_gemma_unstructured,bridges_llama_semi-structured,bridges_llama_structured,bridges_llama_unstructured,bridges_qwen_semi-structured,bridges_qwen_structured,bridges_qwen_unstructured
0,1529451.0,002cefd9f5d64200b06a3af70b332c49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1600006.0,002cefd9f5d64200b06a3af70b332c49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1593387.0,002cefd9f5d64200b06a3af70b332c49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1592477.0,002cefd9f5d64200b06a3af70b332c49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1590316.0,002cefd9f5d64200b06a3af70b332c49,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
for df in final_model_dfs:
    print(len(df))

19117
19117
19117
19117
19117
19117
19117
19117
19117


In [6]:
main_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24057 entries, 0 to 24056
Data columns (total 11 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   humanjobid                     24057 non-null  float64
 1   cvid                           24057 non-null  object 
 2   bridges_gemma_semi-structured  19129 non-null  object 
 3   bridges_gemma_structured       19129 non-null  object 
 4   bridges_gemma_unstructured     19129 non-null  object 
 5   bridges_llama_semi-structured  19129 non-null  object 
 6   bridges_llama_structured       19129 non-null  object 
 7   bridges_llama_unstructured     19129 non-null  object 
 8   bridges_qwen_semi-structured   19129 non-null  object 
 9   bridges_qwen_structured        19129 non-null  object 
 10  bridges_qwen_unstructured      19129 non-null  object 
dtypes: float64(1), object(10)
memory usage: 2.0+ MB


In [7]:
main_df.to_excel("../bridge_table.xlsx")

In [8]:
# 1. Define the master lists based on your pipeline's needs
expected_models = ['qwen', 'gemma', 'llama'] 
# Added 'semi-structured' to match your Traceback error
expected_prompts = ['structured', 'unstructured', 'semi-structured'] 

# 2. Get the full list of all IDs
all_ids = [tuple(i) for i in main_df[['cvid', "humanjobid"]].values]

todo = {}

for model in expected_models:
    todo[model] = {}
    for prompt in expected_prompts:
        # Standardize the label used in columns: "model prompt"
        # We check both "model_prompt" and "model prompt" to be safe
        pair_label = f"{model}_{prompt}"
        bridge_col = f'bridges_{pair_label}'
        
        # Check if we have any existing data for this combination
        if bridge_col in main_df.columns:
            missing_mask = main_df[bridge_col].isna()
            missing_ids = main_df.loc[missing_mask, ['cvid', 'humanjobid']].values
                
            todo[model][prompt] = [tuple(i) for i in missing_ids] # Ensure they are Python ints
        else:
            # If the model/prompt is totally missing, ALL IDs are todos
            todo[model][prompt] = all_ids

# 3. Save to todo.json
with open('todo_bridges.json', 'w') as f:
    json.dump(todo, f)

print(f"todo.json created. Included {len(expected_prompts)} prompt types for {len(expected_models)} models.")

todo.json created. Included 3 prompt types for 3 models.


In [9]:
for k, v in todo.items():
    for l, p in v.items():
        print(k, l, len(p), p[0])

qwen structured 4928 ('002cefd9f5d64200b06a3af70b332c49', 1529451.0)
qwen unstructured 4928 ('002cefd9f5d64200b06a3af70b332c49', 1529451.0)
qwen semi-structured 4928 ('002cefd9f5d64200b06a3af70b332c49', 1529451.0)
gemma structured 4928 ('002cefd9f5d64200b06a3af70b332c49', 1529451.0)
gemma unstructured 4928 ('002cefd9f5d64200b06a3af70b332c49', 1529451.0)
gemma semi-structured 4928 ('002cefd9f5d64200b06a3af70b332c49', 1529451.0)
llama structured 4928 ('002cefd9f5d64200b06a3af70b332c49', 1529451.0)
llama unstructured 4928 ('002cefd9f5d64200b06a3af70b332c49', 1529451.0)
llama semi-structured 4928 ('002cefd9f5d64200b06a3af70b332c49', 1529451.0)
